# 03 – Price distribution across cities

- **Research Question:** How are Airbnb listing prices distributed across major tourist destinations? Are prices normally distributed or heavily right-skewed by luxury listings? Which cities show higher price dispersion, and how do medians compare to arithmetic means when evaluating true market levels?
- **Variables:** `price`
- **Statistical Tools:**
  - **Measures of Central Tendency:** Sample mean vs Sample median.
  - **Measures of Dispersion:** Sample standard deviation, Interquartile Range, and Relative Dispersion.
  - **Measures of Shape:** Fisher-Pearson Skewness and Kurtosis.
  - **Outlier Detection:** Tukey's upper fence rule: $\text{Outlier} > Q_3 + 1.5 \cdot IQR$.
  - **Visualizations:**
    - Distribution histograms with kernel density estimates and explicit mean vs median comparison lines.
    - Boxplots showing medians, quartiles ($Q_1, Q_3$), whiskers, and outlier listings, with a focused head-to-head comparison between Barcelona and Madrid (both in EUR).

In [1]:
from importlib.metadata import distributions
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.cities import CITIES
from src.load import load_all
from src.study_utils import export_table

cols = ['id', 'price', 'room_type', 'latitude', 'longitude', 'neighbourhood_cleansed']
data = load_all(usecols=cols)

data = data.dropna(subset=['price']).copy()

# Adding display names and currency identifiers
data['city_name'] = data['city'].map({k: v.name for k, v in CITIES.items()})
data['currency'] = data['city'].map({k: v.currency for k, v in CITIES.items()})

# Illustrative FX assumptions, not sourced historical exchange rates (EUR per local unit).
fx_to_eur = {
    'EUR': 1.00,
    'USD': 0.86,
    'JPY': 0.0055,
    'MXN': 0.05
}
data['fx_to_eur'] = data['currency'].map(fx_to_eur)
data['price_eur'] = data['price'] * data['fx_to_eur']

num_offers = len(data)
num_cities = data['city'].nunique()
print(f"Loaded {num_offers:,} valid price offers across {num_cities} cities.")

Loaded 154,226 valid price offers across 6 cities.


In [2]:
# Statistical measures of central tendency, dispersion, shape, and outliers
stats_records = []

for city_key, city_obj in CITIES.items():
    s = data[data['city'] == city_key]['price']
    
    # Central tendency
    mean_val = s.mean()
    median_val = s.median()
    
    # Dispersion
    std_val = s.std()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    rel_iqr = (iqr / median_val) * 100
    
    # Shape measures
    skew_val = stats.skew(s)
    kurt_val = stats.kurtosis(s)
    
    # Outliers (Tukey's Upper Fence: Q3 + 1.5 * IQR)
    upper_fence = q3 + 1.5 * iqr
    outliers_cnt = (s > upper_fence).sum()
    outliers_pct = (outliers_cnt / len(s)) * 100
    
    stats_records.append({
        'City': city_obj.name,
        'Currency': city_obj.currency,
        'Offers': len(s),
        'Mean': round(mean_val, 2),
        'Median': round(median_val, 2),
        'Mean / Median': round(mean_val / median_val, 2),
        'Std Dev (s)': round(std_val, 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Rel IQR (%)': round(rel_iqr, 1),
        'Skewness': round(skew_val, 2),
        'Kurtosis': round(kurt_val, 2),
        'Upper Fence (Q3 + 1.5*IQR)': round(upper_fence, 2),
        'Outliers Count': outliers_cnt,
        'Outliers (%)': round(outliers_pct, 2)
    })

#Comprehensive Price Distribution Statistics
df_stats = pd.DataFrame(stats_records).set_index('City')
df_stats

,Currency,Offers,Mean,Median,Mean / Median,Std Dev (s),Q1,Q3,IQR,Rel IQR (%),Skewness,Kurtosis,Upper Fence (Q3 + 1.5*IQR),Outliers Count,Outliers (%)
City,,,,,,,,,,,,,,,
Barcelona,EUR,13355,240.62,177.67,1.35,339.47,89.29,285.50,196.21,110.4,9.04,146.01,579.82,586,4.39
Los Angeles,USD,38186,383.40,223.67,1.71,663.86,115.82,405.98,290.16,129.7,9.29,147.81,841.22,3256,8.53
Madrid,EUR,19172,180.06,125.00,1.44,555.62,83.74,184.00,100.26,80.2,32.24,1462.13,334.38,1369,7.14
New York,USD,21515,278.23,174.69,1.59,531.73,96.88,302.72,205.85,117.8,19.12,701.27,611.49,1608,7.47
Tokyo,JPY,32361,31084.56,19578.00,1.59,65610.35,13466.00,30429.50,16963.50,86.6,25.09,1550.50,55874.75,2442,7.55
Mexico City,MXN,29637,2913.69,1756.00,1.66,9398.89,1073.00,2853.00,1780.00,101.4,65.92,7360.91,5523.00,2461,8.30


In [3]:
# Direct comparison between the two Eurozone cities: Barcelona vs Madrid
bcn_prices = data[data['city'] == 'barcelona']['price']
mad_prices = data[data['city'] == 'madrid']['price']

# Mann-Whitney U test
u_stat, p_val = stats.mannwhitneyu(bcn_prices, mad_prices, alternative='two-sided')

bcn_med = bcn_prices.median()
mad_med = mad_prices.median()
bcn_mean = bcn_prices.mean()
mad_mean = mad_prices.mean()
bcn_iqr = bcn_prices.quantile(0.75) - bcn_prices.quantile(0.25)
mad_iqr = mad_prices.quantile(0.75) - mad_prices.quantile(0.25)
bcn_fence = bcn_prices.quantile(0.75) + 1.5 * bcn_iqr
mad_fence = mad_prices.quantile(0.75) + 1.5 * mad_iqr

spain_comparison = pd.DataFrame([
    {
        'Metric': 'Number of listings',
        'Barcelona': f"{len(bcn_prices):,}",
        'Madrid': f"{len(mad_prices):,}",
        'Difference (BCN vs MAD)': f"{len(bcn_prices) - len(mad_prices):,}"
    },
    {
        'Metric': 'Median Price (€)',
        'Barcelona': f"€{bcn_med:.2f}",
        'Madrid': f"€{mad_med:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_med - mad_med:.2f} (+{(bcn_med / mad_med - 1)*100:.1f}%)"
    },
    {
        'Metric': 'Mean Price (€)',
        'Barcelona': f"€{bcn_mean:.2f}",
        'Madrid': f"€{mad_mean:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_mean - mad_mean:.2f} (+{(bcn_mean / mad_mean - 1)*100:.1f}%)"
    },
    {
        'Metric': 'Interquartile Range - IQR (€)',
        'Barcelona': f"€{bcn_iqr:.2f}",
        'Madrid': f"€{mad_iqr:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_iqr - mad_iqr:.2f}"
    },
    {
        'Metric': 'Upper Outlier Fence (€)',
        'Barcelona': f"€{bcn_fence:.2f}",
        'Madrid': f"€{mad_fence:.2f}",
        'Difference (BCN vs MAD)': f"+€{bcn_fence - mad_fence:.2f}"
    },
    {
        'Metric': 'Mann-Whitney U Test',
        'Barcelona': f"U = {u_stat:,.0f}",
        'Madrid': f"p = {p_val:.2e}",
        'Difference (BCN vs MAD)': f"Two-sided test, p = {p_val:.2e}"
    }
]).set_index('Metric')

spain_comparison

,Barcelona,Madrid,Difference (BCN vs MAD)
Metric,,,
Number of listings,"13,355","19,172","-5,817"
Median Price (€),€177.67,€125.00,+€52.67 (+42.1%)
Mean Price (€),€240.62,€180.06,+€60.56 (+33.6%)
Interquartile Range - IQR (€),€196.21,€100.25,+€95.96
Upper Outlier Fence (€),€579.82,€334.38,+€245.43
Mann-Whitney U Test,"U = 155,985,046",p = 5.17e-247,"Two-sided test, p = 5.17e-247"


In [4]:
# Price Distribution Histograms with Mean vs Median Comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (city_key, city_obj) in enumerate(CITIES.items()):
    ax = axes[idx]
    s = data[data['city'] == city_key]['price']
    
    # Clip at 99th percentile for visualization to clearly display the core distribution
    p99 = s.quantile(0.99)
    s_vis = s[s <= p99]
    
    mean_val = s.mean()
    median_val = s.median()
    skew_val = stats.skew(s)
    kurt_val = stats.kurtosis(s)
    
    sns.histplot(s_vis, kde=True, ax=ax, color='#1f77b4', bins=40, stat='density', alpha=0.45)
    
    # Mean line (Red dashed)
    ax.axvline(mean_val, color='#d62728', linestyle='--', linewidth=2, label=f'Mean: {mean_val:,.1f}')
    
    # Median line (Dark blue solid)
    ax.axvline(median_val, color='#003f5c', linestyle='-', linewidth=2, label=f'Median: {median_val:,.1f}')
    
    ax.set_title(f"{city_obj.name} ({city_obj.currency})\nSkewness = {skew_val:.2f} | Kurtosis = {kurt_val:.1f}", 
                 fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(f"Price ({city_obj.currency})", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(loc='upper right', frameon=True, fontsize=9)

plt.suptitle("Figure 1: Airbnb Price Distributions Across All 6 Cities (Clipped at 99th Percentile)\nDemonstrating Strong Right-Skewness: Mean > Median Everywhere", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study3_price_distribution_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11834/1092119727.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Boxplots and outlier identification
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

city_order = ['Barcelona', 'Madrid', 'Los Angeles', 'New York', 'Tokyo', 'Mexico City']
palette_cities = ['#2b5c8f', '#457b9d', '#e63946', '#d62828', '#f4a261', '#2a9d8f']

# All 6 cities in Standardized EUR Benchmark
sns.boxplot(
    data=data, x='city_name', y='price_eur', order=city_order,
    hue='city_name', legend=False,
    palette=palette_cities, ax=ax1, showfliers=True,
    flierprops=dict(marker='o', markersize=2, alpha=0.25, color='#777777')
)
ax1.set_ylim(0, 1000)
ax1.set_title('Price Boxplots (Illustrative FX assumptions, y <= €1,000)', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Price (EUR equivalent)', fontsize=11)
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=15)

eur_medians = data.groupby('city_name')['price_eur'].median().reindex(city_order)
for idx, (c_name, med) in enumerate(eur_medians.items()):
    ax1.text(idx, med + 15, f"€{med:.0f}", ha='center', va='bottom', fontsize=9.5, fontweight='bold')

# Head-to-Head Comparison: Barcelona vs Madrid (Both native EUR)
spain_data = data[data['city'].isin(['barcelona', 'madrid'])].copy()

sns.boxplot(
    data=spain_data, x='city_name', y='price', order=['Barcelona', 'Madrid'],
    hue='city_name', legend=False,
    palette=['#2b5c8f', '#457b9d'], ax=ax2, width=0.45,
    flierprops=dict(marker='o', markersize=3, alpha=0.35, color='#d62728')
)
ax2.set_ylim(0, 800)
ax2.set_title('Head-to-Head Comparison: Barcelona vs Madrid (Native EUR)', fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Price (EUR)', fontsize=11)
ax2.set_xlabel('')

# Annotate Q1, Median, Q3, and Upper Outlier Fence (Q3 + 1.5 * IQR)
for idx, c_key in enumerate(['barcelona', 'madrid']):
    s = spain_data[spain_data['city'] == c_key]['price']
    q1 = s.quantile(0.25)
    med = s.median()
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    fence = q3 + 1.5 * iqr
    
    # Draw fence line
    ax2.hlines(fence, idx - 0.25, idx + 0.25, color='#d62728', linestyle=':', linewidth=2)
    ax2.text(idx + 0.28, fence, f"Upper Fence: €{fence:.1f}\n($Q_3 + 1.5 \\cdot IQR$)", 
             ha='left', va='center', fontsize=8.5, color='#d62728', fontweight='bold')
    ax2.text(idx, med + 15, f"Median: €{med:.1f}\nIQR: €{iqr:.1f}", 
             ha='center', va='bottom', fontsize=9, fontweight='bold', 
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85))

plt.suptitle("Figure 2: Price Dispersion and Outlier Analysis (Tukey's Fences)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study3_price_boxplots_and_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11834/2861643548.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Findings
All six price distributions are right-skewed and their means exceed their medians. Barcelona has a median of EUR 177.67 versus EUR 125.00 in Madrid (42.1% higher), with IQRs of EUR 196.21 and EUR 100.26. The two-sided Mann–Whitney result is computed in the table; it tests a distributional difference, not the equality of medians alone.

Upper-fence outlier shares are 4.39% in Barcelona and 7.14% in Madrid **among listings with observed prices**. The fence flags unusually high prices relative to each city's distribution; it does not verify luxury status or distinguish correct prices from errors. All observations remain in statistics. Histograms omit the top 1% for display, and boxplot axis limits hide some points.

Illustrative conversion assumptions are EUR 0.86/USD, 0.0055/JPY and 0.05/MXN; no date or primary source was documented for them. Cross-currency figures are scenarios, not measured purchasing power or historical currency comparisons. Barcelona–Madrid uses native EUR. Location, quote timing, stay length and accommodation mix may explain part of the gap; tourism and regulation are not tested as causes.


In [6]:
import plotly.express as px

price_col = 'price_eur' if 'price_eur' in data.columns else 'price'
color_cap = data[price_col].quantile(0.95)

hover_cols = ['room_type']
if 'neighbourhood' in data.columns:
    hover_cols.append('neighbourhood')
elif 'neighbourhood_cleansed' in data.columns:
    hover_cols.append('neighbourhood_cleansed')

fig = px.scatter_geo(
    data,
    lat='latitude',
    lon='longitude',
    color=price_col,
    range_color=[0, color_cap],
    color_continuous_scale='Viridis',
    projection="natural earth",
    hover_name='city_name' if 'city_name' in data.columns else None,
    hover_data=hover_cols,
    title=f"Geographic Price Distribution (Color capped at 95th percentile: €{color_cap:.0f})"
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.write_html('../maps/study03_price_global.html', include_plotlyjs=True)
print('Saved maps/study03_price_global.html')

Saved maps/study03_price_global.html


In [7]:
import plotly.express as px

CITY = "Madrid"
df = data[data["city_name"] == CITY].copy()

price_col = "price_eur" if "price_eur" in df.columns else "price"
color_cap = df[price_col].quantile(0.95)

hover_cols = ["room_type"]
if "neighbourhood_cleansed" in df.columns:
    hover_cols.append("neighbourhood_cleansed")

fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    color=price_col,
    range_color=[0, color_cap],
    color_continuous_scale="Viridis",
    hover_data=hover_cols,
    title=f"Price Distribution in {CITY} (capped at 95th pctl: €{color_cap:.0f})",
    zoom=12,
    center={"lat": df["latitude"].median(), "lon": df["longitude"].median()},
    opacity=0.6,
    map_style="carto-positron",
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.write_html('../maps/study03_price_madrid.html', include_plotlyjs=True)
print('Saved maps/study03_price_madrid.html')


Saved maps/study03_price_madrid.html


### Spatial Interpretation of Price Distribution
- **Spatial Clustering of Luxury:** The map visually confirms our earlier statistical findings regarding right-skewness. The highest-priced listings (bright yellow points representing the top 5%) are not randomly scattered; they form tight geographic clusters strictly concentrated in the city center and prime tourist/coastal areas.
- **Suburban Affordability:** Standard and budget-friendly accommodations (darker points) dominate the peripheral neighborhoods, reflecting a clear distance-to-center price gradient.

In [8]:
export_table(df_stats, 'study03_price_stats', index=True)
export_table(spain_comparison, 'study03_spain_comparison', index=True)
export_table(pd.DataFrame([{'U':u_stat,'p_value':p_val,'n_barcelona':len(bcn_prices),'n_madrid':len(mad_prices),
    'prob_superiority':u_stat/(len(bcn_prices)*len(mad_prices))}]), 'study03_price_test')


,U,p_value,n_barcelona,n_madrid,prob_superiority
0,155985046.5,5.172584e-247,13355,19172,0.609216
